In [3]:
import tensorflow as tf 
import numpy as np
import gym
from tensorflow import keras 
n_inputs = 4 
model = keras.models.Sequential([ 
    keras.layers.Dense(5, activation="elu", input_shape=[n_inputs]), 
    keras.layers.Dense(1, activation="sigmoid"), 
])

REINFORCE算法：

1.首先，让神经网络策略多次参与游戏，然后在每个步骤中计算梯度，使所选择的动作更有可能发生——但不要使用这些梯度。

2.一旦运行了几个回合，就可以计算每个动作的优势

3.如果某个动作的优势为正，则表示该动作可能很好，并且你希望应用较早计算出的梯度来使该动作将来更有可能被选择。但是，如果该动作的优点是负面的，则表示该动作可能是不好的，你希望应用相反的梯度以使该动作在将来被选择的可能性较小。解决方法是简单地把每个梯度向量乘以相应动作的优势。

4.最后，计算所有得到的梯度向量的均值，并使用其执行“梯度下降”步骤。

In [6]:
def play_one_step(env, obs, model, loss_fn):
    with tf.GradientTape() as tape:
        # 修正：确保obs是正确的形状
        left_proba = model(obs[np.newaxis])
        action = (tf.random.uniform([1, 1]) > left_proba)
        y_target = tf.constant([[1.]]) - tf.cast(action, tf.float32)
        loss = tf.reduce_mean(loss_fn(y_target, left_proba))
    grads = tape.gradient(loss, model.trainable_variables)
    
    # 修正：处理action的形状
    action = int(tf.argmax(tf.concat([left_proba, 1 - left_proba], axis=1)[0]))
    
    # 执行一步并返回5个值
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    
    return obs, reward, done, grads

def play_multiple_episodes(env, n_episodes, n_max_steps, model, loss_fn):
    all_rewards = []
    all_grads = []
    
    for episode in range(n_episodes):
        current_rewards = []
        current_grads = []
        
        # 修正：正确处理reset返回值
        obs, info = env.reset()
        
        for step in range(n_max_steps):
            obs, reward, done, grads = play_one_step(env, obs, model, loss_fn)
            current_rewards.append(reward)
            current_grads.append(grads)
            
            if done:
                break
        
        all_rewards.append(current_rewards)
        all_grads.append(current_grads)
    
    return all_rewards, all_grads


def discount_rewards(rewards, discount_factor): 
    discounted = np.array(rewards) 
    for step in range(len(rewards) - 2, -1, -1): 
        discounted[step] += discounted[step + 1] * discount_factor 
    return discounted 

def discount_and_normalize_rewards(all_rewards, discount_factor):
    all_discounted_rewards = []
    for rewards in all_rewards:
        discounted_rewards = []
        cumulative_reward = 0
        for step in reversed(range(len(rewards))):
            cumulative_reward = rewards[step] + cumulative_reward * discount_factor
            discounted_rewards.append(cumulative_reward)
        all_discounted_rewards.append(list(reversed(discounted_rewards)))
    
    # 归一化
    flat_rewards = [reward for rewards in all_discounted_rewards for reward in rewards]
    reward_mean = np.mean(flat_rewards)
    reward_std = np.std(flat_rewards)
    
    all_normalized_discounted_rewards = [
        [(reward - reward_mean) / reward_std for reward in rewards]
        for rewards in all_discounted_rewards
    ]
    return all_normalized_discounted_rewards

In [7]:
discount_rewards([10, 0, -50], discount_factor=0.8), 
discount_and_normalize_rewards([[10, 0, -50], [10, 20]], discount_factor=0.8)

[[-0.2843507149652278, -0.8659771773941027, -1.1891029898545886],
 [1.2666531848451055, 1.072777697368814]]

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(1, activation='sigmoid', input_shape=(4,))  # CartPole-v1有4个观测值
])
env = gym.make("CartPole-v1", render_mode="human")
n_iterations = 150 
n_episodes_per_update = 10 
n_max_steps = 200 
discount_factor = 0.95 
optimizer = keras.optimizers.Adam(lr=0.01) 
loss_fn = keras.losses.binary_crossentropy

for iteration in range(n_iterations): 
    all_rewards, all_grads = play_multiple_episodes( 
        env, n_episodes_per_update, n_max_steps, model, loss_fn) 
    all_final_rewards = discount_and_normalize_rewards(all_rewards, 
                                                      discount_factor) 
    all_mean_grads = [] 
    for var_index in range(len(model.trainable_variables)): 
        mean_grads = tf.reduce_mean( 
            [final_reward * all_grads[episode_index][step][var_index] 
            for episode_index, final_rewards in enumerate(all_final_rewards) 
                for step, final_reward in enumerate(final_rewards)], axis=0) 
        all_mean_grads.append(mean_grads) 
    optimizer.apply_gradients(zip(all_mean_grads, model.trainable_variables))

贝尔曼最优方程

$$
V^*(s) = \max_a \sum_{s'} T(s, a, s') \left[ R(s, a, s') + \gamma \cdot V^*(s') \right], \quad \text{对于所有 } s
$$

在此等式中：

- $ T(s, a, s') $ 是从状态 $ s $ 到状态 $ s' $ 的转移概率，给定智能体选择了行动 $ a $。  

- $ R(s, a, s') $ 是智能体从状态 $ s $ 执行动作 $ a $ 后转移到状态 $ s' $ 所获得的奖励，给定智能体选择了动作 $ a $。  

- $ \gamma $ 是折扣因子。


值迭代算法

$$
V_{k+1}(s) \leftarrow \max_a \sum_{s'} T(s, a, s') \left[ R(s, a, s') + \gamma \cdot V_k(s') \right], \quad \text{对于所有 } s
$$

在此等式中，$ V_k(s) $ 是算法第 $ k $ 次迭代时状态 $ s $ 的估计值。


Q值迭代算法

$$
Q_{k+1}(s, a) \leftarrow \sum_{s'} T(s, a, s') \left[ R(s, a, s') + \gamma \cdot \max_{a'} Q_k(s', a') \right], \quad \text{对所有 } (s', a')
$$

一旦你有了最佳 Q 值，就可以轻松地定义最佳策略，记为 $ \pi^*(s) $：  
当智能体处于状态 $ s $ 时，它应该为该状态选择具有最高 Q 值的动作：

$$
\pi^*(s) = \arg\max_a Q^*(s, a)
$$



In [5]:
transition_probabilities = [ # shape=[s, a, s'] 
        [[0.7, 0.3, 0.0], [1.0, 0.0, 0.0], [0.8, 0.2, 0.0]], 
        [[0.0, 1.0, 0.0], None, [0.0, 0.0, 1.0]], 
        [None, [0.8, 0.1, 0.1], None]] 
rewards = [ # shape=[s, a, s'] 
        [[+10, 0, 0], [0, 0, 0], [0, 0, 0]], 
        [[0, 0, 0], [0, 0, 0], [0, 0, -50]], 
        [[0, 0, 0], [+40, 0, 0], [0, 0, 0]]] 
possible_actions = [[0, 1, 2], [0, 2], [1]]

Q_values = np.full((3, 3), -np.inf) # -np.inf for impossible actions 
for state, actions in enumerate(possible_actions): 
    Q_values[state, actions] = 0.0  # for all possible actions 

gamma = 0.90 # the discount factor 
for iteration in range(50): 
    Q_prev = Q_values.copy() 
    for s in range(3): 
        for a in possible_actions[s]: 
            Q_values[s, a] = np.sum([ 
                    transition_probabilities[s][a][sp] 
                    * (rewards[s][a][sp] + gamma * np.max(Q_prev[sp])) 
                for sp in range(3)]) 
Q_values, np.argmax(Q_values, axis=1)

(array([[18.91891892, 17.02702702, 13.62162162],
        [ 0.        ,        -inf, -4.87971488],
        [       -inf, 50.13365013,        -inf]]),
 array([0, 0, 1], dtype=int64))

TD学习算法会据实际观察到的转变和奖励来更新状态值的估计值

TD学习算法:

$$
V_{k+1}(s) \leftarrow (1 - \alpha) V_k(s) + \alpha \big( r + \gamma \cdot V_k(s') \big)
$$

或等效地：

$$
V_{k+1}(s) \leftarrow V(s) + \alpha \cdot \delta_k(s, r, s')
$$

其中  
$$
\delta_k(s, r, s') = r + \gamma \cdot V_k(s') - V_k(s)
$$

在此等式中：

- $ \alpha $ 是学习率（例如 0.01）。
- $ r + \gamma \cdot V_k(s') $ 被称为 **TD 目标**（TD target）。
- $ \delta_k(s, r, s') $ 被称为 **TD 误差**（TD error）。

---

> **注**：此等式第一种形式的一种更简洁写法是使用符号 $ a \xleftarrow{\alpha} b $，表示  
> $$
> a_{k+1} \leftarrow (1 - \alpha) \cdot a_k + \alpha \cdot b_k
> $$  
> 因此，公式第一行可改写为：
> $$
> V(s) \xleftarrow{\alpha} r + \gamma \cdot V(s')
> $$
> （此处 $ \xleftarrow{\alpha} $ 表示以学习率 $ \alpha $ 向目标值做加权更新）

TD学习与随机梯度下降有很多相似之处，特别是它一次处理一个样本。而且，就像随机梯度下降一样，它只有在逐渐降低学习率的情况下才能真正收敛（否则它将不断在最佳Q值附近震荡）。

对于每个状态s，此算法仅跟踪智能体离开该状态时获得的即时奖励的运行平均值，加上其预期在以后获得的奖励（假设其动作最佳）